# Ohio air quality: interactive Altair charts, part 2

**The idea:** this is a second interactive notebook. The first one (`AQI_Interactive_Altair_Colab`) stays as it is. This one holds the other charts of the long walkthrough that you asked for, and every chart is **interactive**: you can zoom in, move around, hover to read the numbers, and use a menu where a chart has one. The data, the clues and the split are the same as on the poster. Here 13 models are compared, not only the 6 of the poster.

**How to run:** choose **Runtime > Run all**. When the first cell asks, upload the Excel file `AQI_Ohio_2022_2026_with_weather.xlsx`. It takes about 5 to 8 minutes, most of it for training the 13 models. Altair 5 or newer is needed, and so is XGBoost (Colab has both; if a cell says "No module named xgboost", run a new cell with `!pip install xgboost`).

**How to use a chart**
- **Scroll** with the mouse wheel (or two fingers on a trackpad) over a chart to **zoom**.
- **Drag** the chart to **move** it. **Double-click** to reset it (or run its cell again).
- **Hover** over a dot, bar, box or line to read its numbers.
- A chart with a **menu** (the heatmap, the confusion matrix) has the menu on the chart: pick a name and the chart changes.
- On a **bar chart** only the number axis zooms. The names stay in place (Altair cannot zoom a list of names). The charts made of **boxes** (the heatmap and the confusion matrix) cannot zoom for the same reason: hover over a box and use the menu.
- While the mouse is over a chart, the wheel zooms the chart. Move the mouse off the chart to scroll the page.
- To keep a chart as a picture, click the three dots at its top right and choose **Save as PNG**.

**The charts, and where they are in the long walkthrough**

| Here | Charts | In the walkthrough |
|---|---|---|
| Step 2 | The 10 counties with the highest average AQI, and county by month | Step 4 |
| Step 3 | The 7 weather columns, weather against AQI, wind direction | Step 5 |
| Step 7 | Random forest A: more trees, real vs guess | Step 12 |
| Step 8 | Hist Gradient Boosting: error after each tree, real vs guess | Step 14 |
| Step 9 | The ranking of all 13 models (the table and 3 charts) | Step 18 and the model ranking |
| Step 10 | The same 92 days, model by model | Step 19 |
| Step 11 | The confusion matrix, with a menu of models | new |

**Words you will see**
- **AQI:** the Air Quality Index, a number for how polluted the air is. Up to 50 is good, 51 to 100 is moderate, and above 100 is unhealthy (at first for sensitive groups only).
- **Clue:** a fact we know on a day (today's AQI, the weather, the date). The models use the clues to guess tomorrow's AQI.
- **Real AQI:** the AQI that was measured. **Guess:** what a model said the day before.
- **Train days / test days:** a model learns from the train days (2022-2024). We grade it on the test days (2025 to May 2026), which it has never seen.
- **MAE:** how many AQI points the guess is wrong by, on average. **RMSE:** like MAE, but big mistakes count extra. Smaller is better for both.
- **Difference:** test error minus train error. A big positive Difference is the warning sign of overfitting.
- **Tied:** two models whose gap could be just luck (Step 9 explains how we check).
- **Confusion matrix:** a table that counts the days by what the AQI really was and what the model guessed (Step 11).
- **Altair:** the Python library that draws the charts.

**Note:** the numbers in the text come from our run. If your Colab has other library versions, a number can differ in the second decimal, and XGBoost can move a little more.

## Step 1. Get the data

Run the next cell. Click **Choose files** and pick the Excel file. Each row is one county on one day: its AQI, the main pollutant and the weather.

In [ ]:
import numpy as np
import pandas as pd
import altair as alt
from google.colab import files

alt.data_transformers.disable_max_rows()              # Altair stops at 5,000 rows by default. This lifts the limit.
print("Altair version:", alt.__version__)             # these charts need Altair 5 or newer
blue, orange, grey = "#2a78d6", "#eb6834", "#555555"  # the three colors of the charts

uploaded = files.upload()                             # a "Choose files" button appears: pick the Excel file
filename = list(uploaded)[0]                          # the name of the file you uploaded
df = pd.read_excel(filename)                          # read the Excel file into a table called df
df = df.rename(columns={"county Name": "County"})     # a shorter column name
df = df.sort_values(["County", "Date"]).reset_index(drop=True)   # one county after another, oldest day first
print(len(df), "rows,", df.County.nunique(), "counties, from", df.Date.min().date(), "to", df.Date.max().date())

## Step 2. Which counties have the worst air?

**Chart 1** shows the **10 counties with the highest average AQI**, the worst air. Each bar is one county: its average AQI over the whole file. We only rank the counties with at least one year (365 days) of data, because the average of a county with very little data cannot be trusted. Hover over a bar to see the number of days. Zoom in to compare bars that are close. To show more or fewer counties, change `top_n` in the cell.

In [ ]:
counts = df.groupby("County").AQI.agg(["mean", "count"])          # each county's average AQI and its number of days
long_enough = counts[counts["count"] >= 365]                      # keep the counties with at least one year of data
county_means = long_enough.reset_index().rename(columns={"mean": "Average AQI", "count": "Days"})
county_means["Average AQI"] = county_means["Average AQI"].round(1)
print(len(county_means), "counties kept. Left out:", counts[counts["count"] < 365].index.tolist())

top_n = 10                                                         # how many counties to show (try 5 or 15)
top_counties = county_means.nlargest(top_n, "Average AQI")         # the counties with the highest average AQI

chart_counties = alt.Chart(top_counties).mark_bar(color=blue).encode(
    x=alt.X("Average AQI:Q", title="Average AQI, January 2022 to May 2026"),
    y=alt.Y("County:N", sort="-x", title=None),
    tooltip=["County:N", alt.Tooltip("Average AQI:Q", format=".1f"), alt.Tooltip("Days:Q", format=",")],
).properties(title=f"The {top_n} counties with the highest average AQI", width=600, height=30 * top_n).interactive(bind_y=False)

chart_counties

**A check before we read it.** A few counties have no readings from November to February, so their average covers fewer months. The next cell compares all 36 counties (not only the 10 in the chart) on March to October only, which every county has, and checks whether the order changes.

In [ ]:
kept = df[df.County.isin(county_means.County)]                                            # the 36 counties with at least one year of data
nov_to_feb = kept[kept.Date.dt.month.isin([11, 12, 1, 2])].groupby("County").size()       # days with readings from November to February
print(len(county_means) - len(nov_to_feb), "of the", len(county_means), "counties have no readings from November to February")

all_year = kept.groupby("County").AQI.mean().sort_values(ascending=False)                 # the order of Chart 1, for all 36 counties
march_oct = kept[kept.Date.dt.month.between(3, 10)].groupby("County").AQI.mean().sort_values(ascending=False)    # the same counties, March to October only
print("Top 5, all days:", list(all_year.index[:5]), "| March to October only:", list(march_oct.index[:5]))
print("Bottom 2, all days:", list(all_year.index[-2:]), "| March to October only:", list(march_oct.index[-2:]))
moved = (pd.Series(range(len(all_year)), index=all_year.index) - pd.Series(range(len(march_oct)), index=march_oct.index)).abs()    # places a county moves
print(int((moved <= 3).sum()), "of the", len(moved), "counties move by 3 places or fewer")

**What this means**
- The five highest averages are Cuyahoga (56.2), Butler (55.3), Hamilton (54.7), Franklin (53.1) and Lucas (50.3). The chart shows only the top 10, so the lowest are not in it: Belmont (29.8) and Athens (31.9). Cuyahoga is 26.4 points above Belmont.
- Four of the five highest are the counties of Cleveland (Cuyahoga), Cincinnati (Hamilton), Columbus (Franklin) and Toledo (Lucas). So the place matters, which is why a county's usual AQI is one of the 20 clues (Step 5).
- 36 counties have at least a year of data. Columbiana (266 days), Fayette (128) and Harrison (42) are left out.
- **The fairness check above:** 12 of the 36 counties have no readings from November to February, so their average covers the other months only. When all 36 counties are compared on March to October only, the same 5 counties stay on top (Butler and Hamilton swap places), the same 2 stay at the bottom, and 32 of the 36 counties move by 3 places or fewer. So the chart gives a fair picture.

**Chart 2** puts the place and the month together: one box for every county and month. The darker the box, the higher the average AQI. Hover over a box to read its numbers. Use the menu to highlight one county: the other counties fade.

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
df["Month name"] = df.Date.dt.month_name().str[:3]                      # "January" -> "Jan"
big = df[df.County.isin(county_means.County)]                           # only the counties with at least one year of data
heat = big.groupby(["County", "Month name"]).AQI.agg(["mean", "count"]).reset_index()
heat = heat.rename(columns={"mean": "Average AQI", "count": "Days"})    # the average AQI of every county in every month
heat["Average AQI"] = heat["Average AQI"].round(1)
worst = heat.loc[heat.groupby("County")["Average AQI"].idxmax()]        # each county's worst month
print("Worst month, county by county:", worst["Month name"].value_counts().to_dict())

In [ ]:
county_order = list(county_means.sort_values("Average AQI", ascending=False).County)         # the worst county on top
menu = alt.binding_select(options=[None] + county_order, labels=["All counties"] + county_order, name="Highlight a county: ")
pick = alt.selection_point(fields=["County"], bind=menu)                                       # the menu picks one county

chart_heat = alt.Chart(heat).mark_rect().encode(
    x=alt.X("Month name:N", sort=months, axis=alt.Axis(labelAngle=0), title=None),
    y=alt.Y("County:N", sort=county_order, title=None),
    color=alt.Color("Average AQI:Q", scale=alt.Scale(scheme="reds"), title="Average AQI"),
    opacity=alt.condition(pick, alt.value(1), alt.value(0.25)),                                # the other counties fade
    tooltip=["County:N", "Month name:N", alt.Tooltip("Average AQI:Q", format=".1f"), alt.Tooltip("Days:Q", format=",")],
).add_params(pick).properties(title="Average AQI by county and month", width=450, height=600)

chart_heat

**What this means**
- **June is the worst month in 35 of the 36 counties.** The darkest box is Cuyahoga in June (75.1). The lightest is Belmont in September (23.2). The only county whose worst month is not June is Belmont (February, 34.0).
- **June stands out.** Averaged over the counties, June is 59.2, and every other month is between 38.3 (October) and 51.4 (July). About 7% of the county-days in June are above 100, against 1.6% or less in every other month, and 63% of all the days above 100 in the file are in June.
- The rows are in the order of Chart 1, the worst county on top. 12 counties have no boxes from November to February: they sent no data in those months. Use the menu to follow one county across the year.

## Step 3. The weather

The file has 8 weather columns. We use 7 of them as they are. The 8th, the wind direction, is handled at the end of this step.

**Chart 3** has one small chart for each of the 7 weather columns. It counts the days in 20 groups of values (3,000 random days are enough to see the shape). Each small chart zooms on its own: scroll over it, and hover over a bar to see its group and its number of days.

In [ ]:
weather = ["Temp Max (°F)", "Temp Mean (°F)", "Precipitation (in)", "Wind Speed Max (mph)",
           "Wind Gust Max (mph)", "Relative Humidity (%)", "Solar Radiation (MJ/m²)"]       # the 7 weather columns
sample = df.sample(3000, random_state=0)                                  # 3,000 random days

small = []
for number, column in enumerate(weather):                                 # one small chart per weather column
    days, edges = np.histogram(sample[column], bins=20)                   # count the days in 20 groups of values
    bars = pd.DataFrame({"From": edges[:-1], "To": edges[1:], "Days": days, "Zero": 0})      # "Zero" is the bottom of every bar
    zoom = alt.selection_interval(bind="scales", encodings=["x", "y"], name=f"zoom{number}")      # a zoom of its own for this small chart
    small.append(alt.Chart(bars).mark_bar(color=blue).encode(
        x=alt.X("From:Q", title=column), x2="To:Q", y=alt.Y("Days:Q", title=None), y2="Zero:Q",
        tooltip=[alt.Tooltip("From:Q", format=".1f"), alt.Tooltip("To:Q", format=".1f"), "Days:Q"],
    ).properties(width=150, height=110).add_params(zoom))
chart_weather = alt.concat(*small, columns=4).properties(title="How many days have each value? (3,000 random days)")

chart_weather

**What this means**
- **Rain** is the most lopsided: 2,289 of the 3,000 days (76%) are in the first group (up to 0.12 inch), and only 58 days have more than 1 inch. A few very wet days stretch the chart out to 2.3 inches. Zoom in on the right end to see them.
- **Temperature** has its tallest bars on the warm side (the daily high peaks at 80 to 85 °F) and a long tail of cold days down to 8 °F.
- **Wind speed** and **gusts** have a long tail on the right: the tallest bars are at 7 to 8 mph and 20 to 23 mph, but the biggest days reach 37 and 70 mph.
- **Humidity** sits in the middle: 72% of the days are between 60% and 85%. **Sunshine** is the most spread out: no bar holds more than 218 of the 3,000 days (7%).

**Chart 4** shows the **correlation** between the AQI and each weather column on the same day. A correlation goes from -1 to +1: positive means the two tend to rise together, negative means one tends to be high when the other is low, and 0 means no straight-line relation.

In [ ]:
correlation = df[["AQI"] + weather].corr()["AQI"].drop("AQI")              # +1: move together, 0: unrelated, -1: move opposite
correlation = correlation.rename("Correlation").rename_axis("Weather").reset_index()
correlation["Effect"] = (correlation.Correlation > 0).map({True: "higher AQI", False: "lower AQI"})
correlation["Correlation"] = correlation.Correlation.round(2)
print(correlation)

chart_correlation = alt.Chart(correlation).mark_bar().encode(
    x=alt.X("Correlation:Q", title="Correlation with the AQI of the same day"),
    y=alt.Y("Weather:N", sort="-x", title=None),
    color=alt.Color("Effect:N", title="A higher value means", scale=alt.Scale(domain=["higher AQI", "lower AQI"], range=[orange, blue])),
    tooltip=["Weather:N", "Correlation:Q", "Effect:N"],
).properties(title="Which weather goes with a higher or lower AQI?", width=480, height=240).interactive(bind_y=False)

chart_correlation

**What this means**
- **Sunshine (+0.35), the daily high (+0.30) and the mean temperature (+0.27)** go with a higher AQI on the same day. **Wind gusts (-0.23), wind speed (-0.20), humidity (-0.17) and rain (-0.14)** go with a lower AQI.
- **All 7 links are weak** (the largest is 0.35). For comparison, today's AQI and tomorrow's AQI have a link of 0.63 on the train days. So the weather alone does not decide the AQI, and today's AQI is a much stronger clue than any weather column. (The first notebook compared the clues with tomorrow's AQI, so its numbers are a little different.)
- **A link is not a cause.** Part of the link is the time of year: June is warm and sunny and has the worst air. When we compare days of the same month only, the links of sunshine and temperature shrink (sunshine 0.35 to 0.27, daily high 0.30 to 0.26, mean temperature 0.27 to 0.21), while the links of wind gusts (-0.22), wind speed (-0.17), humidity (-0.19) and rain (-0.16) stay about the same.
- **A likely reason, not shown by this chart:** ozone is the pollutant that sets the AQI on 62% of the days above 100 (287 of 460), and ozone is made in the air when sunshine and heat act on the exhaust of traffic and industry (general knowledge, not tested here).

**Chart 5:** the wind **direction** is in degrees (0 = north, 90 = east, 180 = south, 270 = west). We group it into 8 compass directions and draw the average AQI of each. Zoom in on the number axis to see the small differences.

In [ ]:
compass = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
degrees = df["Wind Direction (°)"]                                        # 0 to 360 degrees
direction = pd.Series(compass)[((degrees + 22.5) // 45 % 8).astype(int)].to_numpy()       # one of 8 compass directions
by_direction = df.groupby(direction).AQI.agg(["mean", "count"]).reset_index()
by_direction.columns = ["Direction", "Average AQI", "Days"]
by_direction["Average AQI"] = by_direction["Average AQI"].round(1)

chart_wind = alt.Chart(by_direction).mark_bar(color=blue).encode(
    x=alt.X("Direction:N", sort=compass, axis=alt.Axis(labelAngle=0), title="Wind direction"),
    y=alt.Y("Average AQI:Q"),
    tooltip=["Direction:N", alt.Tooltip("Average AQI:Q", format=".1f"), alt.Tooltip("Days:Q", format=",")],
).properties(title="Average AQI for each wind direction", width=450, height=240).interactive(bind_x=False)

chart_wind

**What this means**
- Winds from the **southeast (49.2) and south (49.1)** go with the highest average AQI, then southwest (47.0) and east (46.8). Winds from the **west (40.9) and northwest (41.0)** go with the lowest.
- The spread is 8.3 points. That is smaller than the gap between counties (26.4, Chart 1) and between June and October (20.9, Chart 2), so the wind direction separates the days less than the place and the month do.
- The most common directions are southwest (11,163 days) and west (9,083 days). Hover over a bar to see the number of days.
- **A link, not a proof.** The chart mixes all counties and all months, so it cannot tell whether a wind really brings dirty air to a place. For the models the direction is turned into two numbers (the sine and the cosine of the angle), so that 359 degrees and 1 degree count as neighbors (Step 4).

## Step 4. The answer and the clues

For every day we want to guess **tomorrow's AQI**. That is the **answer**. The **clues** are facts we know on that day:
- today's AQI, the AQI 1, 2 and 6 days ago, and the averages of the last 3 and the last 7 days;
- the time of year, the weekday of tomorrow, whether ozone set today's AQI, and the wind direction;
- 7 weather columns and the county's usual AQI (the last clue is added in Step 5).

Every calculation stays inside one county, so "yesterday" never comes from another county.

In [ ]:
weather = ["Temp Max (°F)", "Temp Mean (°F)", "Precipitation (in)", "Wind Speed Max (mph)",
           "Wind Gust Max (mph)", "Relative Humidity (%)", "Solar Radiation (MJ/m²)"]       # the 7 weather columns

each_county = df.groupby("County")                    # every calculation below stays inside one county

df["tomorrow"] = each_county.AQI.shift(-1)            # the ANSWER: the AQI on the next row
df["aqi_today"] = df.AQI
df["aqi_yesterday"] = each_county.AQI.shift(1)        # the AQI on the row before
df["aqi_2_days_ago"] = each_county.AQI.shift(2)
df["aqi_6_days_ago"] = each_county.AQI.shift(6)
df["aqi_3_day_average"] = each_county.AQI.transform(lambda day: day.rolling(3).mean())   # today and the 2 days before
df["aqi_7_day_average"] = each_county.AQI.transform(lambda day: day.rolling(7).mean())   # today and the 6 days before

In [ ]:
angle = 2 * np.pi * df.Date.dt.dayofyear / 365.25                  # the day of the year as an angle on a circle
df["day_of_year_sin"] = np.sin(angle)
df["day_of_year_cos"] = np.cos(angle)
df["tomorrow_weekday"] = (df.Date + pd.Timedelta(days=1)).dt.dayofweek          # 0 = Monday ... 6 = Sunday
df["ozone_main"] = (df["Defining Parameter"] == "Ozone").astype(int)            # 1 if ozone set today's AQI

wind_angle = np.deg2rad(df["Wind Direction (°)"])                  # the wind direction is also an angle on a circle
df["wind_dir_sin"] = np.sin(wind_angle)
df["wind_dir_cos"] = np.cos(wind_angle)

In [ ]:
early_clues = ["aqi_today", "aqi_yesterday", "aqi_2_days_ago", "aqi_6_days_ago", "aqi_3_day_average", "aqi_7_day_average",
               "day_of_year_sin", "day_of_year_cos", "tomorrow_weekday", "ozone_main", "wind_dir_sin", "wind_dir_cos"]
clues_without_county = early_clues + weather       # 19 clues. The 20th, county_average, needs the split of Step 5.

# Keep only the days that have every clue, and where the next row really is the next day
next_row_is_tomorrow = (each_county.Date.shift(-1) - df.Date).dt.days == 1
df = df[next_row_is_tomorrow].dropna(subset=clues_without_county + ["tomorrow"])
print("Days we can use:", len(df))

## Step 5. Train days and test days

The models learn from the **train days** (before 2025). We check them on the **test days** (2025 to May 2026), which they never see while learning. The 20th clue, the county's usual AQI, is worked out from the train days only.

In [ ]:
train = df[df.Date < "2025-01-01"].copy()        # 2022-2024: the computer learns from these days
test = df[df.Date >= "2025-01-01"].copy()        # 2025 to May 2026: we check its guesses on these days

usual = train.groupby("County").tomorrow.mean()           # each county's usual AQI, from the training days only
train["county_average"] = train.County.map(usual)
test["county_average"] = test.County.map(usual)           # the test days get the training days' numbers: no peeking

clues = early_clues + ["county_average"] + weather        # the 20 clues, in the same order as on the poster
X_train, y_train = train[clues], train.tomorrow           # X = the clues, y = the answers
X_test, y_test = test[clues], test.tomorrow
print("Training days:", len(train), "| Test days:", len(test))

## Step 6. Train 13 models and let them guess

Thirteen models learn from the **train days only**. Then each one guesses every day, train days and test days. The first cells below are the models you know from the poster and from the long walkthrough. The others were added to compare more kinds of models.
- **Baseline:** the lazy guess "tomorrow = today". It learns nothing.
- **Ridge, Lasso and the straight-line SVM** are straight-line models. The clues are first put on the same scale, with the scale learned from the train days only. Lasso can switch clues off. The SVM ignores small misses while it learns.
- **Curvy SVM:** a support vector machine with a curved rule (the "RBF" kind). Its 3 settings were chosen on 2024, inside the train days, and never on the test days.
- **Random forest A** (100 trees, default settings), **random forest B** (the poster's forest: 300 trees, at most 12 questions deep, at least 10 days in every leaf, each question looks at half of the clues) and the **tuned random forest** (the poster's tuned one).
- **Hist Gradient Boosting** and **XGBoost:** trees built one after another, each one correcting the mistakes of the trees before it.
- **AdaBoost** is a third way of building trees one after another. Its default settings do badly, so we also try gentler settings. Those settings were picked after looking at the test scores, so that score is a little flattering.
- **SARIMAX stand-in:** one straight-line model for each county, from today's AQI, yesterday's AQI, the 7-day average, the weather and the month of the year. It is **not** a real SARIMAX (a time-series model). It only stands in for it, so read its score as a rough guide.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import LinearSVR, SVR

scaler = StandardScaler().fit(X_train)                        # learn the scale from the train days only
Z_train, Z_test = scaler.transform(X_train), scaler.transform(X_test)
scaled = {"Ridge": Ridge(alpha=1.0),
          "Lasso": Lasso(alpha=0.1),
          "Straight-line SVM": LinearSVR(C=1.0, epsilon=0.0, max_iter=20000, dual="auto", random_state=0),
          "Curvy SVM": SVR(kernel="rbf", C=3, gamma=0.006, epsilon=0.5, cache_size=1500)}     # this one takes a minute or two
for name, model in scaled.items():
    model.fit(Z_train, y_train)                               # learn from the train days only
    train[name] = model.predict(Z_train)                      # a guess for every train day
    test[name] = model.predict(Z_test)                        # a guess for every test day
    print(name, "done")

In [ ]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor                              # if this says "No module named xgboost", run a new cell with: !pip install xgboost

trees = {"Random forest A": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
         "Random forest B": RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=10, max_features=0.5, random_state=42, n_jobs=-1),
         "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42),
         "XGBoost": XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42),
         "AdaBoost (defaults)": AdaBoostRegressor(random_state=42),
         "AdaBoost (gentler)": AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=5, min_samples_leaf=20), n_estimators=100, learning_rate=0.05, random_state=42)}
for name, model in trees.items():
    model.fit(X_train, y_train)                               # learn from the train days only
    train[name] = model.predict(X_train)                      # a guess for every train day
    test[name] = model.predict(X_test)                        # a guess for every test day (days it has never seen)
    print(name, "done")

In [ ]:
tuned_clues = [c for c in clues if c != "county_average"] + ["county_average"]    # the poster's order: county_average is the last clue
by_date = train.sort_values(["Date", "County"])                                    # the poster fitted this forest on the train days in date order
tuned = RandomForestRegressor(n_estimators=400, max_depth=16, min_samples_leaf=10, max_features=0.7, random_state=42, n_jobs=-1)
tuned.fit(by_date[tuned_clues], by_date.tomorrow)
train["Tuned random forest"], test["Tuned random forest"] = tuned.predict(train[tuned_clues]), tuned.predict(test[tuned_clues])

train["Baseline"], test["Baseline"] = train.aqi_today, test.aqi_today            # the lazy guess: tomorrow = today
print("Tuned random forest and baseline done")

In [ ]:
from sklearn.linear_model import LinearRegression

for frame in (train, test):                                       # the month of the year, as a point on a circle
    frame["month_sin"] = np.sin(2 * np.pi * frame.Date.dt.month / 12)
    frame["month_cos"] = np.cos(2 * np.pi * frame.Date.dt.month / 12)
line_clues = ["aqi_today", "aqi_yesterday", "aqi_7_day_average", "month_sin", "month_cos"] + weather

train["SARIMAX stand-in"], test["SARIMAX stand-in"] = np.nan, np.nan
for county_name, days in train.groupby("County"):                 # one straight-line model for each county
    line = LinearRegression().fit(days[line_clues], days.tomorrow)
    train.loc[days.index, "SARIMAX stand-in"] = line.predict(days[line_clues])
    days_test = test[test.County == county_name]
    if len(days_test):                                            # some counties have no test days
        test.loc[days_test.index, "SARIMAX stand-in"] = line.predict(days_test[line_clues])

In [ ]:
models = ["Baseline", "Ridge", "Lasso", "Straight-line SVM", "Curvy SVM", "SARIMAX stand-in", "Random forest A", "Random forest B",
          "Tuned random forest", "Hist Gradient Boosting", "XGBoost", "AdaBoost (defaults)", "AdaBoost (gentler)"]
print(len(models), "models. Missing guesses on the train days:", int(train[models].isna().sum().sum()), "| on the test days:", int(test[models].isna().sum().sum()))

## Step 7. Random forest A (the default settings)

A **decision tree** asks a chain of yes/no questions about the clues ("is today's AQI above 52?", "is the wind gust above 30?") and ends in a **leaf**. Its guess is the average of the train days that ended in that leaf. A **random forest** grows many trees, each one learning from a different random sample of the days, and **averages their guesses**. Many different trees make different mistakes, and averaging cancels part of them.

**Forest A** uses 100 trees and the default settings, which let every tree grow until its leaves are tiny.

**Chart 6** gives the test RMSE when the forest is made of only its first tree, its first 2 trees, and so on up to all 100. Hover over the line to read the number of trees and the RMSE. Zoom in on the first 20 trees.

In [ ]:
from sklearn.metrics import mean_squared_error

forest_a = trees["Random forest A"]                                                       # the forest trained in Step 6
each_tree = np.array([tree.predict(X_test.to_numpy()) for tree in forest_a.estimators_])  # one row of guesses per tree
growing = np.cumsum(each_tree, axis=0) / np.arange(1, len(each_tree) + 1)[:, None]        # the average of the first 1, 2, 3, ... trees
rmse_by_trees = ((growing - y_test.to_numpy()) ** 2).mean(axis=1) ** 0.5
curve = pd.DataFrame({"Trees": range(1, len(each_tree) + 1), "RMSE": rmse_by_trees.round(3)})
print("Test RMSE with 1 tree:", round(rmse_by_trees[0], 2), "| 10 trees:", round(rmse_by_trees[9], 2), "| 100 trees:", round(rmse_by_trees[-1], 2))
print("RMSE of the whole forest on the TRAIN days:", round(mean_squared_error(y_train, train["Random forest A"]) ** 0.5, 2))

chart_trees = alt.Chart(curve).mark_line(point=True, color=blue).encode(
    x=alt.X("Trees:Q", title="Number of trees in the forest"),
    y=alt.Y("RMSE:Q", title="Test RMSE (smaller is better)", scale=alt.Scale(zero=False)),
    tooltip=["Trees:Q", alt.Tooltip("RMSE:Q", format=".2f")],
).properties(title="Random forest A: more trees, better guesses", width=560, height=300).interactive()

chart_trees

**What this means**
- **One tree is poor, a few trees are already good.** One tree has an RMSE of 17.28, worse than the lazy baseline (14.57). With 3 trees the forest beats the baseline (14.22), and with 10 trees it reaches 12.85.
- **Most of the gain comes early.** From 1 to 10 trees the RMSE falls by 4.43. From 10 to 50 trees it falls by 0.60 (to 12.25). From 50 to 100 trees it falls by only 0.08 (to 12.17).
- **More trees do not hurt.** The line flattens, and it never goes up by more than 0.005 from one tree to the next. So more trees cost time, not accuracy.
- **But the forest has memorized the train days.** Its RMSE is 3.76 on the train days and 12.17 on the test days: a Difference of +8.41, the largest of the 13 models (Step 9). Averaging many trees keeps the test error small anyway.

**Chart 7:** each dot is one test day. Along the bottom is the real AQI of tomorrow, up the side is the guess. A perfect guess sits on the black line. We draw 1,500 random days with an AQI up to 100, and **every** one of the rare days above 100 (in red). Hover over a dot to see the county and the day, and zoom in on the crowded part.

In [ ]:
bad = y_test > 100                                                  # the rare bad days: tomorrow's real AQI is above 100
shown = y_test[~bad].sample(1500, random_state=0).index.union(y_test[bad].index)     # 1,500 random other days + EVERY bad day
day_type = pd.Series(np.where(bad, "Above 100", "Up to 100"), index=y_test.index)
day_colors = alt.Scale(domain=["Up to 100", "Above 100"], range=[blue, "#d62728"])   # blue for most days, red for the bad days
perfect = pd.DataFrame({"Real AQI": [0, 250], "Guess": [0, 250]})                   # a perfect guess would sit on this line

dots = pd.DataFrame({"County": test.County[shown], "Day": test.Date[shown] + pd.Timedelta(days=1), "Real AQI": y_test[shown],
                     "Guess": test["Random forest A"][shown].round(1), "Real AQI is": day_type[shown]})
points = alt.Chart(dots).mark_circle(size=20, opacity=0.4).encode(
    x="Real AQI:Q", y="Guess:Q", color=alt.Color("Real AQI is:N", scale=day_colors),
    tooltip=["County:N", alt.Tooltip("Day:T", format="%b %d, %Y"), "Real AQI:Q", "Guess:Q"])
line = alt.Chart(perfect).mark_line(color="black").encode(x="Real AQI:Q", y="Guess:Q")

chart_scatter_a = (points + line).properties(title="Random forest A: real AQI vs guess", width=440, height=440).interactive()
chart_scatter_a

**What this means**
- The dots follow the black line, so the forest has learned something real. But the cloud is wide: on average the guess is 8.71 points away from the real AQI (the MAE of Step 9).
- **The guesses stay near the middle.** On the 446 test days with a real AQI of 20 or less, the average guess is 31.9. The 109 days above 100 (the red dots) have an average real AQI of 118.2 but an average guess of 69.2. All 109 red dots are below the black line, and the highest guess for any of them is 98.6. The forest never called a really bad day "above 100".
- A model that is unsure plays safe and guesses near the middle. The rare bad days (109 of the 14,596 test days, 0.75%) are the hardest to see coming.

## Step 8. Hist Gradient Boosting

**Boosting** is a different kind of ensemble. The trees are built **one after another**: the first tree makes a rough guess, the second tree tries to fix the mistakes of the first, the third fixes what is still wrong, and so on. ("Hist" means that the values of each clue are first grouped into bins, which makes it fast.) We use the default settings: up to 100 small trees.

**Chart 8** shows the error after 1 tree, after 2 trees, and so on, on the train days and on the test days. Hover over a line to read the numbers.

In [ ]:
boost = trees["Hist Gradient Boosting"]                                                 # the booster trained in Step 6
train_curve = [mean_squared_error(y_train, guess) ** 0.5 for guess in boost.staged_predict(X_train)]    # the error after each new tree
test_curve = [mean_squared_error(y_test, guess) ** 0.5 for guess in boost.staged_predict(X_test)]
curve = pd.DataFrame({"Trees": list(range(1, len(test_curve) + 1)) * 2,
                      "RMSE": np.round(train_curve + test_curve, 3),
                      "Days": ["Train days"] * len(train_curve) + ["Test days"] * len(test_curve)})
print(f"{boost.n_iter_} trees were built. The test error is lowest after tree {int(np.argmin(test_curve)) + 1} (RMSE {min(test_curve):.2f}). After the last tree it is {test_curve[-1]:.2f}.")

chart_boost = alt.Chart(curve).mark_line(point=True).encode(
    x=alt.X("Trees:Q", title="Number of trees built so far"),
    y=alt.Y("RMSE:Q", title="RMSE (smaller is better)", scale=alt.Scale(zero=False)),
    color=alt.Color("Days:N", sort=["Train days", "Test days"], scale=alt.Scale(domain=["Train days", "Test days"], range=[orange, blue]), title=None),
    tooltip=["Days:N", "Trees:Q", alt.Tooltip("RMSE:Q", format=".2f")],
).properties(title="Hist Gradient Boosting: error after each tree", width=560, height=300).interactive()

chart_boost

**What this means**
- Both lines start high and drop fast. The first tree alone gives an RMSE of 16.93 on the train days and 15.59 on the test days. After 10 trees both are at 13.04.
- **Then the lines split.** The train line keeps falling (9.85 at tree 64, 9.26 at the last tree). The test line flattens: from tree 30 on it stays within 0.1 of its lowest value. The gap between the two lines at the end is the Difference: +2.85.
- **The last trees do not help.** In our run the lowest test RMSE comes after tree 64 (12.08), and after the last tree it is 12.12. The rise is tiny (0.03), but it shows that the trees after 64 do not help on the test days.
- **Do not pick the number of trees from this chart.** The best point is only visible on the test days. Choosing it would use the test days to tune the model, and the test score would become flattering. The fair way is to choose the number of trees inside the train days. Hist Gradient Boosting can do this by itself (it holds back a part of the train days and stops when they stop improving), and in our run it did not stop before 100 trees.

**Chart 9** is the same dot chart as in Step 7 (real AQI against guess), for Hist Gradient Boosting.

In [ ]:
dots = pd.DataFrame({"County": test.County[shown], "Day": test.Date[shown] + pd.Timedelta(days=1), "Real AQI": y_test[shown],
                     "Guess": test["Hist Gradient Boosting"][shown].round(1), "Real AQI is": day_type[shown]})
points = alt.Chart(dots).mark_circle(size=20, opacity=0.4).encode(
    x="Real AQI:Q", y="Guess:Q", color=alt.Color("Real AQI is:N", scale=day_colors),
    tooltip=["County:N", alt.Tooltip("Day:T", format="%b %d, %Y"), "Real AQI:Q", "Guess:Q"])
line = alt.Chart(perfect).mark_line(color="black").encode(x="Real AQI:Q", y="Guess:Q")

chart_scatter_boost = (points + line).properties(title="Hist Gradient Boosting: real AQI vs guess", width=440, height=440).interactive()
chart_scatter_boost

**What this means**
- The picture is like forest A. The dots follow the black line, but the 109 red dots (the days above 100) all sit below it: their average real AQI is 118.2 and their average guess is 66.9. Only 1 of the 109 gets a guess above 100 (105.1).
- Forest A and Hist Gradient Boosting are alike here: both stay near the middle, and neither is good at the rare bad days (forest A flags 0 of the 109, Hist Gradient Boosting 1). A model trained to keep the average miss small does not take risks.

## Step 9. The ranking of all 13 models

Here are all 13 models on the same test days, with the same 20 clues, from the best test RMSE to the worst. The table also gives the MAE, the RMSE on the train days, and the **Difference** (test RMSE minus train RMSE): a big positive number is the sign of overfitting.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

rows = [{"Model": name,
         "Test RMSE": mean_squared_error(y_test, test[name]) ** 0.5,
         "Test MAE": mean_absolute_error(y_test, test[name]),
         "Train RMSE": mean_squared_error(y_train, train[name]) ** 0.5} for name in models]
board = pd.DataFrame(rows).sort_values("Test RMSE").reset_index(drop=True)        # the best model first
board["Difference"] = board["Test RMSE"] - board["Train RMSE"]                      # test error minus train error
board.insert(0, "Rank", board.index + 1)                                            # 1 = the smallest RMSE
board["MAE rank"] = board["Test MAE"].rank(method="min").astype(int)                # the rank we get with MAE instead of RMSE
board.round(2)

**What this means**
- **The best models are about 17% better than the lazy baseline.** In our run the tuned forest and forest B both have an RMSE of 12.03, against 14.57 for the baseline (the MAE is 8.57 and 8.56, against 10.27).
- **A straight-line model gets most of the gain.** Going from the baseline to Ridge, Lasso or the straight-line SVM (12.27 to 12.34) gains about 2.2 to 2.3 RMSE points. Going from there to the best model gains only 0.2 to 0.3 more (2% to 3%).
- **Forest A has by far the biggest Difference** (+8.41: 3.76 on the train days, 12.17 on the test days). The Difference is negative for the straight-line models and the baseline: the test days are a little easier for everybody (the baseline, which learns nothing, has an RMSE of 15.29 on the train days and 14.57 on the test days).
- **AdaBoost with the default settings (18.08) is worse than the lazy baseline (14.57).** The gentler settings give 12.55, but we chose them after looking at the test scores, so that number is a little flattering.
- **MAE gives a slightly different order.** In our run the curvy SVM has the smallest MAE (8.50). A likely reason: its training counts every miss the same, while the forests are trained on squared misses, which count big misses extra, like RMSE.
- **XGBoost:** look at its row yourself. Its numbers depend on the version of XGBoost in your Colab, so this text does not describe it.

**Are two models really different?** On 14,596 test days, two models that differ by 0.05 might differ just by luck. To check, we draw the test days again **5,000 times**: each time we pick two-week blocks of days (all counties together) at random, with repeats, and work out every model's RMSE on that new set of days. We keep the gap between each model and the best one. If, in some reruns, the gap is 0 or below, the model is **tied** with the best. If the gap stays above 0 in more than 97 reruns out of 100, the model is **clearly behind**.

In [ ]:
day_number = (test.Date - test.Date.min()).dt.days.to_numpy()          # 0 for the first test day, 1 for the next, ...
block = day_number // 14                                               # two-week blocks of days (all counties together)
n_blocks = int(block.max()) + 1
days_in_block = np.bincount(block, minlength=n_blocks)
squared_miss = {m: np.bincount(block, weights=(test[m].to_numpy() - y_test.to_numpy()) ** 2, minlength=n_blocks) for m in models}

rng = np.random.default_rng(0)                                         # a fixed seed: every run gives the same numbers
draws = np.stack([np.bincount(rng.integers(0, n_blocks, n_blocks), minlength=n_blocks) for _ in range(5000)])    # how often each block is drawn
rmse_runs = {m: np.sqrt(draws @ squared_miss[m] / (draws @ days_in_block)) for m in models}     # every model's RMSE in every rerun
best = board.Model[0]
gap = {m: rmse_runs[m] - rmse_runs[best] for m in models}              # in every rerun: the model's RMSE minus the best model's RMSE
board["Gap to best"] = board["Test RMSE"] - board["Test RMSE"][0]
board["Gap low"] = [np.percentile(gap[m], 2.5) for m in board.Model]   # 95 reruns out of 100 land between low and high
board["Gap high"] = [np.percentile(gap[m], 97.5) for m in board.Model]
board["Group"] = np.where(board["Gap low"] <= 0, "Tied with the best", "Clearly behind")
board[["Rank", "Model", "Gap to best", "Gap low", "Gap high", "Group"]].round(3)

**Chart 10** shows the same ranking as bars: the RMSE on the left and the MAE on the right, in the same order. The blue bars are the models tied with the best, the grey ones are clearly behind. All the bars start at 0, so the gaps between the top models are small: zoom in on the right end of a chart to see them (each chart zooms on its own). Hover over a bar for all the numbers of the model.

In [ ]:
model_order = list(board.Model)                                        # best model first
group_colors = alt.Scale(domain=["Tied with the best", "Clearly behind"], range=[blue, "#b0b0b0"])
tip = ["Model:N", "Group:N", alt.Tooltip("Test RMSE:Q", format=".2f"), alt.Tooltip("Test MAE:Q", format=".2f"),
       alt.Tooltip("Train RMSE:Q", format=".2f"), alt.Tooltip("Difference:Q", format="+.2f")]
zoom_rmse = alt.selection_interval(bind="scales", encodings=["x"], name="zoom_rmse")     # each chart has a zoom of its own (number axis only)
zoom_mae = alt.selection_interval(bind="scales", encodings=["x"], name="zoom_mae")

rmse_bars = alt.Chart(board).mark_bar().encode(
    x=alt.X("Test RMSE:Q", title="RMSE: big misses count extra"), y=alt.Y("Model:N", sort=model_order, title=None),
    color=alt.Color("Group:N", scale=group_colors, title="Compared with the best"), tooltip=tip,
).properties(width=260, height=340).add_params(zoom_rmse)
mae_bars = alt.Chart(board).mark_bar(color=grey).encode(
    x=alt.X("Test MAE:Q", title="MAE: average miss"), y=alt.Y("Model:N", sort=model_order, axis=None), tooltip=tip,
).properties(width=260, height=340).add_params(zoom_mae)

chart_ranking = alt.hconcat(rmse_bars, mae_bars).properties(title="How far off is each model on the test days? (smaller is better)")
chart_ranking

**Chart 11** puts the error on the train days next to the error on the test days. The gap between the two bars of a model is its Difference.

In [ ]:
both = board[["Model", "Train RMSE", "Test RMSE", "Difference"]].rename(columns={"Train RMSE": "Train days", "Test RMSE": "Test days"})
both = both.melt(["Model", "Difference"], var_name="Error on", value_name="RMSE")

chart_overfit = alt.Chart(both).mark_bar().encode(
    x=alt.X("RMSE:Q"), y=alt.Y("Model:N", sort=model_order, title=None),
    yOffset=alt.YOffset("Error on:N", sort=["Train days", "Test days"]),                       # two bars for each model
    color=alt.Color("Error on:N", sort=["Train days", "Test days"], scale=alt.Scale(domain=["Train days", "Test days"], range=[orange, blue]), title=None),
    tooltip=["Model:N", "Error on:N", alt.Tooltip("RMSE:Q", format=".2f"), alt.Tooltip("Difference:Q", format="+.2f")],
).properties(title="Error on the train days and on the test days", width=520, height=440).interactive(bind_y=False)

chart_overfit

**What this means**
- Look at the orange bar (train days) and the blue bar (test days) of each model. An orange bar much shorter than the blue one means that the model memorized the train days.
- **Forest A is the clearest case:** 3.76 on the train days and 12.17 on the test days (Difference +8.41). The tuned forest (8.43 and 12.03, Difference +3.60), Hist Gradient Boosting (9.26 and 12.12, +2.85) and forest B (9.35 and 12.03, +2.68) memorize less.
- **Ridge, Lasso, both SVMs, the SARIMAX stand-in, the baseline and the default AdaBoost have a negative Difference** (-0.17 to -0.72): their blue bar is shorter than their orange bar. They are too simple to memorize, and the test days are a little easier: the baseline, which learns nothing, has an RMSE of 15.29 on the train days and 14.57 on the test days.
- **A big Difference does not mean a bad model.** In our run the three best models by test RMSE (the tuned forest, forest B and Hist Gradient Boosting) all have a positive Difference (+2.7 to +3.6). The blue bar decides the ranking. The Difference is the warning light.

**Chart 12** is the tie check: for every model except the best one, the dot is the gap to the best model in our run and the line is the range of 95 reruns out of 100. The dashed black line is 0. A line that crosses it means "tied". The last models are far to the right, so scroll to zoom in on the left end, where the top models are.

In [ ]:
rest = board[board.Rank > 1]                                           # every model except the best one
gap_title = f"RMSE gap to the best model ({best})"
line_range = alt.Chart(rest).mark_rule(strokeWidth=3).encode(
    x=alt.X("Gap low:Q", title=gap_title), x2="Gap high:Q", y=alt.Y("Model:N", sort=model_order, title=None),
    color=alt.Color("Group:N", scale=group_colors, legend=None))
dot = alt.Chart(rest).mark_circle(size=90, opacity=1).encode(
    x=alt.X("Gap to best:Q", title=gap_title), y=alt.Y("Model:N", sort=model_order, title=None),
    color=alt.Color("Group:N", scale=group_colors, title="Compared with the best"),
    tooltip=["Model:N", "Group:N", alt.Tooltip("Gap to best:Q", format="+.3f"), alt.Tooltip("Gap low:Q", format="+.3f"), alt.Tooltip("Gap high:Q", format="+.3f")])
zero = alt.Chart(pd.DataFrame({"Gap to best": [0]})).mark_rule(color="black", strokeDash=[4, 3]).encode(x=alt.X("Gap to best:Q", title=gap_title))

chart_gap = (line_range + dot + zero).properties(title="Is the gap to the best model real?", width=560, height=340).interactive(bind_y=False)
chart_gap

**What this means**
- The tuned forest is the best of our run, but forest B is only 0.0015 behind, so they can swap places in another run.
- **Tied with the best (the line crosses 0):** forest B (-0.03 to +0.03), Hist Gradient Boosting (-0.05 to +0.22) and the curvy SVM (-0.03 to +0.27). On other days any of them could have come first. So there is no single winner, there is a top group.
- **Clearly behind (the whole line is right of 0):** forest A (+0.05 to +0.23), Lasso (+0.09 to +0.40), Ridge (+0.12 to +0.42), the straight-line SVM (+0.15 to +0.50), the SARIMAX stand-in (+0.18 to +0.50) and the gentler AdaBoost (+0.36 to +0.69). They are behind, but only a little (at most 0.7). The baseline (+2.05 to +3.05) and the default AdaBoost (+4.89 to +7.36) are far behind.
- **A limit of this check:** it asks "would other days change the order?", but it only uses days from the same test period. It cannot tell if the order would hold in other years.

## Step 10. The same 92 days, model by model

One county (Cuyahoga, Cleveland): the guesses made on each of the 92 days from 1 June to 31 August 2025, drawn on the day each guess is for (2 June to 1 September). In every small chart the grey line is the real AQI and the red line is the model's guess. The small charts are in the order of the ranking, the best model first, and they all use the same AQI scale (0 to 160), so they can be compared.

**Chart 13:** the small charts **zoom together**. Scroll over any of them to zoom in on the same days in all 13, and drag to move. Hover over a line to read the day and the AQI.

In [ ]:
county = "Cuyahoga"                                                # Cleveland. Change this to any county you like.
window = test[(test.County == county) & (test.Date >= "2025-06-01") & (test.Date < "2025-09-01")]       # June to August 2025

pieces = [pd.DataFrame({"Model": name, "Day": window.Date + pd.Timedelta(days=1),      # a guess made today is for tomorrow
                        "Real AQI": window.tomorrow, "Guess": window[name]}) for name in model_order]
panels = pd.concat(pieces).melt(["Model", "Day"], var_name="Line", value_name="AQI")  # a real line and a guess line for every model
panels["AQI"] = panels.AQI.round(1)
print(county, "has", window.Date.nunique(), "days in the window. Highest real AQI:", int(window.tomorrow.max()),
      "| highest guess of any model except the baseline:", round(window[[m for m in models if m != "Baseline"]].max().max(), 1))

In [ ]:
zoom = alt.selection_interval(bind="scales", encodings=["x"])             # one zoom for all the small charts: they show the same days
lines = alt.Chart(panels).mark_line(strokeWidth=1.5).encode(
    x=alt.X("Day:T", title=None, axis=alt.Axis(format="%b %d", tickCount=3)), y=alt.Y("AQI:Q", scale=alt.Scale(domain=[0, 160])),
    color=alt.Color("Line:N", scale=alt.Scale(domain=["Real AQI", "Guess"], range=["#888888", "#d62728"]), title=None))
hover = alt.Chart(panels).mark_circle(size=60, opacity=0).encode(          # invisible dots: they only show the numbers when you hover
    x=alt.X("Day:T", title=None), y="AQI:Q", tooltip=["Model:N", alt.Tooltip("Day:T", format="%b %d, %Y"), "Line:N", "AQI:Q"])

chart_92days = alt.layer(lines, hover).add_params(zoom).properties(width=210, height=110).facet(
    alt.Facet("Model:N", sort=model_order, title=None), columns=3)
chart_92days

**What this means** (Cuyahoga, June to August 2025)
- **Every model's guess is calmer than reality.** The grey line has big peaks and dips. The red lines of the models (except the Baseline) swing much less. The highest real AQI is 150, on 15 July.
- **The Baseline is the real line, one day late.** Its guess for a day is today's AQI, so its red line is the grey line moved by one day: it reaches 150 on 16 July, when the real AQI had already dropped to 76. This is why the models have to beat it.
- **No top model sees the peak coming.** For 15 July the tuned forest guessed 94, forest B 89, Hist Gradient Boosting 82 and the curvy SVM 74. For the two other days above 100 (23 July: 112, and 4 August: 106) these four guessed between 55 and 69.
- This is one county and 92 days, to show how the models behave. It is not a score. The scores in Step 9 use all the counties and all 14,596 test days.

## Step 11. The confusion matrix

A **confusion matrix** counts the test days by two things: what tomorrow's AQI **really was** (the rows) and what the model **guessed** (the columns). We use the three groups of the poster: 0-50 (good), 51-100 (moderate) and above 100. A model that is always right has all its days on the diagonal, from the top left to the bottom right. The number in each box is a number of days, and the **color** is the share of the real days of that row: dark means a large part of that row's days.

**Chart 14:** use the **menu** to switch from one model to another. Hover over a box to read the days and the share.

In [ ]:
labels = ["0-50 (good)", "51-100 (moderate)", "Above 100"]
group = lambda aqi: pd.cut(aqi, [-np.inf, 50, 100, np.inf], labels=labels)         # which of the 3 groups an AQI is in
real_group = group(y_test)

pieces = []
for name in models:
    table = pd.crosstab(real_group, group(test[name])).reindex(index=labels, columns=labels, fill_value=0)    # days: real group x guessed group
    one = table.stack().rename("Days").reset_index()
    one.columns = ["Real AQI", "Guess", "Days"]
    one["Model"] = name
    pieces.append(one)
matrix = pd.concat(pieces, ignore_index=True)
matrix["Share"] = matrix.Days / matrix.groupby(["Model", "Real AQI"], observed=True).Days.transform("sum")      # the share of the real days of that row
print(matrix[matrix.Model == "Tuned random forest"].pivot(index="Real AQI", columns="Guess", values="Days").loc[labels, labels])

In [ ]:
menu = alt.binding_select(options=models, name="Model: ")
pick_model = alt.selection_point(fields=["Model"], bind=menu, value=[{"Model": "Tuned random forest"}])      # starts with the poster's tuned forest

base = alt.Chart(matrix).encode(
    x=alt.X("Guess:N", sort=labels, axis=alt.Axis(labelAngle=0, orient="top"), title="What the model guessed for tomorrow"),
    y=alt.Y("Real AQI:N", sort=labels, title="What tomorrow's AQI really was"))
boxes = base.mark_rect().encode(color=alt.Color("Share:Q", scale=alt.Scale(scheme="blues", domain=[0, 1]), title="Share of the real days"),
    tooltip=["Model:N", "Real AQI:N", "Guess:N", alt.Tooltip("Days:Q", format=","), alt.Tooltip("Share:Q", format=".0%")])
numbers = base.mark_text(fontSize=16).encode(text=alt.Text("Days:Q", format=","),
    color=alt.condition("datum.Share > 0.5", alt.value("white"), alt.value("black")))

chart_matrix = (boxes + numbers).add_params(pick_model).transform_filter(pick_model).properties(title="Confusion matrix: days by real and guessed group", width=460, height=280)
chart_matrix

**What this means**
- **Start with the tuned forest** (the menu starts on it). Of the 9,772 test days that were really 0-50, it guessed 0-50 on 8,559 (88%). Of the 4,715 days that were really 51-100, it got 2,947 right (63%), but it called 1,765 of them "good air" (37%). In all, 79% of the 14,596 test days land in the right group.
- **The bottom row is the weak spot.** Of the 109 days above 100, the tuned forest called 103 "moderate" and 6 "good". It never said "above 100" for a really bad day (0 of 109). The curvy SVM and forest A also catch 0 of the 109, and Hist Gradient Boosting catches 1.
- **The models play safe.** A model that tries to keep the average miss small guesses near the middle, so it rarely says "above 100". Pick the **Baseline** in the menu: because it copies today's AQI, it catches 16 of the 109 bad days, but it says "above 100" on 108 days, so 92 of those are false alarms. The tuned forest says it on only 4 days, and none of them is right.
- **The cut lines are harsh.** A real AQI of 51 that is guessed as 49 counts as wrong. Of the 1,765 days that were really 51-100 but guessed 0-50 by the tuned forest, 1,257 (71%) had a real AQI of 51 to 60, just over the line.
- **If the goal is a warning for bad days, the RMSE of Step 9 is not enough.** Look at this matrix too.

## Summary in plain English

- **Where and when:** Cuyahoga, Butler, Hamilton, Franklin and Lucas have the worst air on average (56.2 down to 50.3), Belmont and Athens the best (29.8 and 31.9). June is the worst month in 35 of the 36 counties.
- **Weather:** sunshine and warmth go with a higher AQI, wind, humidity and rain with a lower AQI. All the links are weak (the largest is 0.35), so no single weather column decides the AQI.
- **Forest A:** more trees help a lot up to about 10, a little up to about 50, and almost nothing after that (12.25 at 50 trees, 12.17 at 100). It memorizes the train days (Difference +8.41) but still scores well on the test days.
- **Hist Gradient Boosting:** the test error stops improving around tree 30 while the train error keeps falling.
- **The ranking:** the best models are about 17% better than the lazy baseline. In our run the tuned forest, forest B, Hist Gradient Boosting and the curvy SVM are tied with the best. The straight-line models are only 2% to 3% behind. The default AdaBoost is worse than the baseline.
- **Bad days:** on the 109 test days above 100, the top models almost never say "above 100" (0 or 1 of the 109), and in the 92-day chart no model sees the peak of 150 coming. If bad days matter most, look at the confusion matrix and not only at the RMSE.
- **Limits:** the 92-day chart is one county. The test days are one period (2025 to May 2026). The SARIMAX stand-in is not a real SARIMAX. The gentler AdaBoost settings were chosen after looking at the test scores. The XGBoost row depends on your version of XGBoost.